# Visualize a Galewsky shallow-water run

Reads one resampled run from the Zarr dataset produced by
`datagen/galewsky/scripts/postprocess.py` and renders a few snapshots of
the balanced jet, the height perturbation, and the vorticity field.

Dataset layout: `(time, field, lat, lon)` with
`field ∈ {u_phi, u_theta, h, vorticity}`.

## Environment

Needs `xarray`, `zarr`, `numpy`, `matplotlib`. All ship with the `datagen`
project env. Add `ipykernel` / `jupyterlab` if they aren't installed:

```bash
uv add --project datagen ipykernel jupyterlab
uv run --project datagen jupyter lab
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# Point this at any run_XXXX.zarr produced by the sweep.
DATA_PATH = Path("/scicore/home/dokman0000/GROUP/PDEDatasets/SphericalPDEs/galewsky-sw/processed/run_0101.zarr")

ds = xr.open_zarr(DATA_PATH)
ds

In [ ]:
# Quick parameter + shape summary.
print("Dims:", dict(ds.dims))
print("Time range:", float(ds.time.min()) / 3600, "..", float(ds.time.max()) / 3600, "h")
print("Lat range: ", float(ds.lat.min()), "..", float(ds.lat.max()), "deg")
print("Lon range: ", float(ds.lon.min()), "..", float(ds.lon.max()), "deg")
print("Parameters:")
for k, v in ds.attrs.items():
    if k.startswith("param_"):
        print(f"  {k[6:]:11s} = {v}")

## Initial condition sanity check

At `t=0`, expect a compact zonal jet in `u_phi` centered at `lat_center`,
`u_theta ≈ 0` except where the Gaussian height bump is breaking symmetry,
a balanced height drop across the jet in `h`, and vorticity concentrated
in two thin bands on either side of the jet.

In [ ]:
def plot_field(ax, arr, lat, lon, title, cmap="RdBu_r", symmetric=False):
    vmax = np.nanmax(np.abs(arr)) if symmetric else None
    vmin = -vmax if symmetric else None
    im = ax.pcolormesh(lon, lat, arr, cmap=cmap, vmin=vmin, vmax=vmax, shading="auto")
    ax.set_title(title)
    ax.set_xlabel("lon [deg]")
    ax.set_ylabel("lat [deg]")
    plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)


t0 = 0
fig, axes = plt.subplots(2, 2, figsize=(14, 7), constrained_layout=True)
lat = ds.lat.values
lon = ds.lon.values
plot_field(axes[0, 0], ds.fields.sel(field="u_phi").isel(time=t0).values, lat, lon,
           "u_phi (zonal) [m/s]", cmap="RdBu_r", symmetric=True)
plot_field(axes[0, 1], ds.fields.sel(field="u_theta").isel(time=t0).values, lat, lon,
           "u_theta (southward) [m/s]", cmap="RdBu_r", symmetric=True)
plot_field(axes[1, 0], ds.fields.sel(field="h").isel(time=t0).values, lat, lon,
           "h (height perturbation) [m]", cmap="RdBu_r", symmetric=True)
plot_field(axes[1, 1], ds.fields.sel(field="vorticity").isel(time=t0).values, lat, lon,
           "relative vorticity [1/s]", cmap="RdBu_r", symmetric=True)
fig.suptitle(f"t = {float(ds.time.isel(time=t0)) / 3600:.1f} h")
plt.show()

## Time evolution of the vorticity field

The barotropic instability should roll up into a sequence of cyclones
along the northern flank of the jet — the signature Galewsky result.
Six panels evenly spaced through the trajectory.

In [ ]:
n_times = ds.sizes["time"]
panels = np.linspace(0, n_times - 1, 6, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(18, 8), constrained_layout=True, sharex=True, sharey=True)
vort_all = ds.fields.sel(field="vorticity").values
vmax = float(np.nanmax(np.abs(vort_all))) * 0.7  # clip tails for visibility
for ax, ti in zip(axes.flat, panels):
    im = ax.pcolormesh(lon, lat, vort_all[ti], cmap="RdBu_r",
                       vmin=-vmax, vmax=vmax, shading="auto")
    ax.set_title(f"t = {float(ds.time.isel(time=ti)) / 3600:.1f} h")
    ax.set_xlabel("lon [deg]")
    ax.set_ylabel("lat [deg]")
fig.colorbar(im, ax=axes, fraction=0.015, pad=0.02, label="vorticity [1/s]")
fig.suptitle("Relative vorticity evolution — Galewsky barotropic instability")
plt.show()

## Height perturbation over time

Complementary view: `h` visualizes the balanced geostrophic height drop
(strong meridional stripes from the jet balance) overlaid with the
growing perturbation wave train.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8), constrained_layout=True, sharex=True, sharey=True)
h_all = ds.fields.sel(field="h").values
vmax = float(np.nanmax(np.abs(h_all)))
for ax, ti in zip(axes.flat, panels):
    im = ax.pcolormesh(lon, lat, h_all[ti], cmap="RdBu_r",
                       vmin=-vmax, vmax=vmax, shading="auto")
    ax.set_title(f"t = {float(ds.time.isel(time=ti)) / 3600:.1f} h")
    ax.set_xlabel("lon [deg]")
    ax.set_ylabel("lat [deg]")
fig.colorbar(im, ax=axes, fraction=0.015, pad=0.02, label="h [m]")
fig.suptitle("Height perturbation evolution")
plt.show()

## Zonal-mean Hovmöller diagrams

Zonal mean over longitude as a function of `(time, lat)`. Before the
instability breaks the zonal symmetry this is the full story; afterward
the zonally-averaged jet drift and meridional heat transport are visible.

In [ ]:
u_phi_zm = ds.fields.sel(field="u_phi").mean("lon")  # (time, lat)
vort_zm = ds.fields.sel(field="vorticity").mean("lon")

fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
t_h = ds.time.values / 3600.0

vmax_u = float(np.abs(u_phi_zm).max())
im0 = axes[0].pcolormesh(t_h, lat, u_phi_zm.T, cmap="RdBu_r",
                          vmin=-vmax_u, vmax=vmax_u, shading="auto")
axes[0].set_title("Zonal-mean u_phi [m/s]")
axes[0].set_xlabel("time [h]")
axes[0].set_ylabel("lat [deg]")
fig.colorbar(im0, ax=axes[0])

vmax_v = float(np.abs(vort_zm).max()) * 0.8
im1 = axes[1].pcolormesh(t_h, lat, vort_zm.T, cmap="RdBu_r",
                          vmin=-vmax_v, vmax=vmax_v, shading="auto")
axes[1].set_title("Zonal-mean relative vorticity [1/s]")
axes[1].set_xlabel("time [h]")
axes[1].set_ylabel("lat [deg]")
fig.colorbar(im1, ax=axes[1])

plt.show()

## Globally-integrated diagnostics

Area-weighted RMS of `h` and global max of `|u|` over time — smooth
monotonic curves mean the solver is behaving; sudden kinks or explosions
signal instability.

In [ ]:
lat_rad = np.deg2rad(ds.lat.values)
w = np.cos(lat_rad)
w = w / w.sum()

h = ds.fields.sel(field="h").values
u_phi = ds.fields.sel(field="u_phi").values
u_theta = ds.fields.sel(field="u_theta").values
speed = np.sqrt(u_phi ** 2 + u_theta ** 2)

rms_h = np.sqrt((h ** 2).mean(axis=-1) @ w)          # over (lat, lon)
max_speed = speed.max(axis=(-1, -2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(t_h, rms_h)
axes[0].set_xlabel("time [h]")
axes[0].set_ylabel("RMS h [m]")
axes[0].set_title("Area-weighted RMS height perturbation")
axes[1].plot(t_h, max_speed)
axes[1].set_xlabel("time [h]")
axes[1].set_ylabel("max |u| [m/s]")
axes[1].set_title("Global max speed")
plt.show()

## Vorticity animation (MP4)

Writes an MP4 of the vorticity field via `imageio.v3` with `libx264`.
This bundles its own ffmpeg binary through `imageio-ffmpeg`, so no
system-level `ffmpeg` is needed. One-time install into the datagen env:

```bash
uv add --project datagen 'imageio[ffmpeg]'
```

Same pattern as `warpspeed/scripts/make_rollout_video.py` — render each
frame to an RGB array by drawing the figure and grabbing the canvas
buffer, then hand the stack to `iio.imwrite`.

In [ ]:
from pathlib import Path

import imageio.v3 as iio

vort_all = ds.fields.sel(field="vorticity").values
vmax = float(np.nanmax(np.abs(vort_all))) * 0.7
fps = 12
dpi = 100
output_path = Path("galewsky_vorticity.mp4")

frames = []
for i in range(vort_all.shape[0]):
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True, dpi=dpi)
    mesh = ax.pcolormesh(lon, lat, vort_all[i], cmap="RdBu_r",
                          vmin=-vmax, vmax=vmax, shading="auto")
    ax.set_title(f"t = {float(ds.time.isel(time=i)) / 3600:.1f} h")
    ax.set_xlabel("lon [deg]")
    ax.set_ylabel("lat [deg]")
    fig.colorbar(mesh, ax=ax, label="vorticity [1/s]")

    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    # libx264 requires even frame dimensions; crop one pixel if odd.
    frame = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)
    frame = frame[: h - (h % 2), : w - (w % 2), :3]
    frames.append(frame)
    plt.close(fig)

iio.imwrite(
    str(output_path),
    np.stack(frames, axis=0),
    fps=fps,
    codec="libx264",
    pixelformat="yuv420p",
)
print(f"Wrote {output_path}  ({len(frames)} frames @ {fps} fps)")

### Inline playback (no ffmpeg needed)

If you'd rather view the animation inside the notebook without writing
a file, use matplotlib's HTML5 JavaScript writer — it doesn't invoke
ffmpeg or Pillow at all.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
mesh = ax.pcolormesh(lon, lat, vort_all[0], cmap="RdBu_r",
                      vmin=-vmax, vmax=vmax, shading="auto")
title = ax.set_title("t = 0.0 h")
ax.set_xlabel("lon [deg]")
ax.set_ylabel("lat [deg]")
fig.colorbar(mesh, ax=ax, label="vorticity [1/s]")

def update(i):
    mesh.set_array(vort_all[i].ravel())
    title.set_text(f"t = {float(ds.time.isel(time=i)) / 3600:.1f} h")
    return mesh, title

anim = animation.FuncAnimation(fig, update, frames=vort_all.shape[0], interval=80, blit=False)
plt.close(fig)  # suppress the static first frame above the player
HTML(anim.to_jshtml())

In [ ]:
import cartopy.crs as ccrs
from tqdm import trange

# Assuming ds, lon, lat, and vort_all are already defined in your script
vort_all = ds.fields.sel(field="vorticity").values
vmax = float(np.nanmax(np.abs(vort_all))) * 0.7
fps = 12
dpi = 100
output_path = Path("galewsky_vorticity_spherical.mp4")

frames = []
num_frames = vort_all.shape[0]

for i in trange(num_frames):
    fig = plt.figure(figsize=(8, 8), constrained_layout=True, dpi=dpi)
    
    # Optional: Slowly rotate the globe over the course of the animation.
    # For a fixed view, just set cent_lon = 0.0
    cent_lon = (i / num_frames) * 360.0 - 180.0
    
    # Create a spherical (Orthographic) projection
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.Orthographic(
        central_longitude=cent_lon, 
        central_latitude=30.0 # Tilted slightly to see the northern hemisphere better
    ))
    
    ax.set_global()
    ax.gridlines(alpha=0.3, linestyle='--') # Adds faint lat/lon lines
    
    # The crucial part: we tell pcolormesh that the input data is in standard lat/lon degrees
    # using transform=ccrs.PlateCarree()
    mesh = ax.pcolormesh(
        lon, lat, vort_all[i], 
        cmap="RdBu_r",
        vmin=-vmax, vmax=vmax, 
        shading="auto",
        transform=ccrs.PlateCarree()
    )
    
    ax.set_title(f"Galewsky Vorticity | t = {float(ds.time.isel(time=i)) / 3600:.1f} h")
    
    # Adjust colorbar for the new layout
    fig.colorbar(mesh, ax=ax, shrink=0.6, label="vorticity [1/s]", pad=0.05)

    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    
    # libx264 requires even frame dimensions; crop one pixel if odd.
    frame = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)
    frame = frame[: h - (h % 2), : w - (w % 2), :3]
    frames.append(frame)
    plt.close(fig)

iio.imwrite(
    str(output_path),
    np.stack(frames, axis=0),
    fps=fps,
    codec="libx264",
    pixelformat="yuv420p",
)
print(f"Wrote {output_path}  ({len(frames)} frames @ {fps} fps)")